In [6]:
import os
import pandas as pd
def get_chr_names():
    # Mouse chromosomes: 1-19, X, Y, M
    chrs = [f"chr{i}" for i in range(1, 20)]  # chr1 to chr19
    chrs += ["chrX", "chrY", "chrM"]
    return chrs
chr_names = get_chr_names()

# Get merged ME mat from matriline pipeline
mm39_cpg_dir = "/mnt/thomas/matriline/results/GRCm39/bismark_genome/CpGs_location"

def ref_stats(ref_cpg_dir):
    results = []
    for chr_name in chr_names:
        df_cpg_path = os.path.join(ref_cpg_dir, f"CpGslocation_{chr_name}.csv")
        if not os.path.exists(df_cpg_path):
            print(f"File not found: {df_cpg_path}")
            continue
        
        df_cpg = pd.read_csv(df_cpg_path)  # This is a DataFrameGroupBy

        n_cpgs = len(df_cpg)

        results.append({
            "chromosome": chr_name,
            "n_cpgs": n_cpgs,
            })
    cpg_stats = pd.DataFrame(results)
    return cpg_stats

cpg_stats_df = ref_stats(mm39_cpg_dir)
cpg_stats_df.to_csv("/mnt/thomas/matriline/results/GRCm39/bismark_genome/nCpGs_per_chr.csv", index=False)

In [ ]:
import os
import math
import pandas as pd

FASTA_PATH = "/mnt/thomas/matriline/results/GRCm39/mm39.fa"
FAI_PATH = FASTA_PATH + ".fai"

# Your existing CpG location directory
mm39_cpg_dir = "/mnt/thomas/matriline/results/GRCm39/bismark_genome/CpGs_location"

def get_chr_names():
    chrs = [f"chr{i}" for i in range(1, 20)]
    chrs += ["chrX", "chrY", "chrM"]
    return chrs

chr_names = get_chr_names()

def read_fai(fai_path: str):
    """
    Returns dict: chr -> (length, offset, line_bases, line_width)
    line_width includes newline.
    """
    idx = {}
    with open(fai_path, "r") as f:
        for line in f:
            if not line.strip():
                continue
            name, length, offset, line_bases, line_width = line.rstrip("\n").split("\t")[:5]
            idx[name] = (int(length), int(offset), int(line_bases), int(line_width))
    return idx

def count_contexts_both_strands_from_fai(fasta_path: str, fai_index: dict, chr_name: str):
    """
    Counts CpG/CHG/CHH cytosines on BOTH strands in a single forward pass.

    Plus-strand contexts (cytosine at position i where base is 'C'):
      CpG: C followed by G  -> "CG" at i..i+1
      CHG: C H G -> "C[ACT]G" at i..i+2
      CHH: C H H -> "C[ACT][ACT]" at i..i+2

    Minus-strand cytosines correspond to forward-strand 'G' positions.
    Their reverse context is determined by upstream bases on the forward strand:
      CpG (minus): 'G' preceded by 'C' -> "...CG" (counts the opposite-strand cytosine)
      CHG (minus): 'G' with (i-2)=='C' and (i-1) in {A,G,T} -> "C[AGT]G"
      CHH (minus): 'G' with (i-2) in {A,G,T} and (i-1) in {A,G,T} -> "[AGT][AGT]G"
    """
    if chr_name not in fai_index:
        raise ValueError(f"{chr_name} not found in FAI index")

    length, offset, line_bases, _line_width = fai_index[chr_name]
    n_lines = math.ceil(length / line_bases)

    H_plus = set("ACT")      # H for plus-strand contexts (not G)
    H_minus = set("AGT")     # complements of plus-strand H (not C)

    n_C_bases = 0            # forward strand 'C' bases only (simple base count)

    n_CpG_cyt = 0
    n_CHG_cyt = 0
    n_CHH_cyt = 0

    # rolling window for forward scanning
    prev2 = None  # base i-2
    prev1 = None  # base i-1

    with open(fasta_path, "rb") as f:
        f.seek(offset)

        for _ in range(n_lines):
            line = f.readline()
            if not line:
                break
            s = line.strip().decode("ascii").upper()
            if not s:
                continue

            for b in s:
                if b not in ("A", "C", "G", "T"):
                    # treat N/others as breakpoints for context
                    prev2, prev1 = prev1, b
                    continue

                # base count (forward strand only)
                if b == "C":
                    n_C_bases += 1

                # MINUS-strand contexts counted at forward 'G' positions
                if b == "G" and prev1 is not None:
                    if prev1 == "C":
                        n_CpG_cyt += 1  # minus CpG cytosine
                    if prev2 is not None:
                        if prev2 == "C" and prev1 in H_minus:
                            n_CHG_cyt += 1  # minus CHG cytosine
                        elif prev2 in H_minus and prev1 in H_minus:
                            n_CHH_cyt += 1  # minus CHH cytosine

                # PLUS-strand CpG counted at 'C' positions (requires next base = current b)
                if prev1 == "C" and b == "G":
                    n_CpG_cyt += 1  # plus CpG cytosine

                # PLUS-strand CHG/CHH counted when we now have (prev2, prev1, b)
                if prev2 == "C" and prev1 in H_plus:
                    if b == "G":
                        n_CHG_cyt += 1
                    elif b in H_plus:
                        n_CHH_cyt += 1

                prev2, prev1 = prev1, b

    return {
        "n_C_bases": n_C_bases,
        "n_CpG": n_CpG_cyt,
        "n_CHG": n_CHG_cyt,
        "n_CHH": n_CHH_cyt,
    }

def ref_stats(ref_cpg_dir: str, fasta_path: str, fai_path: str):
    fai_index = read_fai(fai_path)

    results = []
    for chr_name in chr_names:
        # count CpG rows from your CpGslocation file (as you had)
        df_cpg_path = os.path.join(ref_cpg_dir, f"CpGslocation_{chr_name}.csv")
        if not os.path.exists(df_cpg_path):
            print(f"File not found: {df_cpg_path}")
            continue

        df_cpg = pd.read_csv(df_cpg_path)
        n_cpg_rows = len(df_cpg)

        # reference context stats from FASTA/FAI
        ctx = count_contexts_both_strands_from_fai(fasta_path, fai_index, chr_name)

        results.append({
            "chromosome": chr_name,
            "n_cpg_rows_in_csv": n_cpg_rows,  # your existing metric
            "n_C_bases": ctx["n_C_bases"],    # forward-strand C bases
            "n_CpG": ctx["n_CpG"],            # CpG cytosines (both strands)
            "n_CHG": ctx["n_CHG"],            # CHG cytosines (both strands)
            "n_CHH": ctx["n_CHH"],            # CHH cytosines (both strands)
        })

    return pd.DataFrame(results)

if __name__ == "__main__":
    cpg_stats_df = ref_stats(mm39_cpg_dir, FASTA_PATH, FAI_PATH)
    out_csv = "/mnt/thomas/matriline/results/GRCm39/bismark_genome/ref_context_stats_per_chr.csv"
    cpg_stats_df.to_csv(out_csv, index=False)
    print(f"Wrote: {out_csv}")

    # Example: quickly inspect chrM
    print(cpg_stats_df[cpg_stats_df["chromosome"] == "chrM"])
